# 판정기 학습 (Colab T4)

준비: `factcheck-judge-colab.zip` 을 내 드라이브 최상단(`MyDrive/`)에 올린다. 런타임 → 런타임 유형 변경 → **T4 GPU**.

위에서부터 차례로 실행. 체크포인트·결과는 `MyDrive/factcheck-judge-runs/` 에 저장되니 연결이 끊겨도 **1·2번 셀 다시 실행 후 3번 셀의 `--resume`** 으로 이어서 한다.

T4 는 bf16 이 없어 `--fp16`(가중치 float32 + fp16 혼합정밀).

In [ ]:
# 1. 드라이브 연결 + 코드·데이터 풀기 + 패키지
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/fj && mkdir -p /content/fj /content/drive/MyDrive/factcheck-judge-runs
!unzip -q /content/drive/MyDrive/factcheck-judge-colab.zip -d /content/fj
!ln -sfn /content/drive/MyDrive/factcheck-judge-runs /content/fj/runs
!pip -q install 'transformers>=5.0' 'peft>=0.13' 'accelerate>=1.0'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 작업 폴더
%cd /content/fj/scripts

In [ ]:
# 3. 학습 (끊긴 뒤 이어서 할 때는 끝에 --resume 추가)
!python train.py --base Qwen/Qwen3-0.6B --name q06b-v4 --fp16

In [ ]:
# 4. 주장 단위 평가(29·18건) — 결과는 runs/q06b-v4/eval.json 에도 저장
!python evaluate.py --base Qwen/Qwen3-0.6B --adapter ../runs/q06b-v4/adapter

In [ ]:
# 5. 어댑터 압축 → MyDrive/factcheck-judge-runs/q06b-v4-adapter.zip (Claude 가 드라이브에서 받아 간다)
!cd ../runs && rm -f q06b-v4-adapter.zip && zip -qr q06b-v4-adapter.zip q06b-v4/adapter q06b-v4/metrics.json q06b-v4/eval.json && ls -l q06b-v4-adapter.zip